# 成绩与主成分

行是学生、列是两次测验。先中心化，再找变化最多的方向。

依赖：NumPy、Matplotlib。按顺序运行，先在纸上预测再核对；图内使用英文标签。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 先计算平均值

两次测验都用百分制。减平均后，第三行是什么？


In [ ]:
data = np.array([[72.,71.],[71.,72.],[69.,68.],[68.,69.]])
mean = data.mean(axis=0)
X = data-mean
print('平均:',mean,'中心化:',X)
assert np.allclose(mean,[70,70])
assert np.allclose(X.T @ X,[[10,8],[8,10]])


## 一个坐标，能恢复多少？

先手算共同方向，再检查 SVD 的方向与重建。方向整体变号不会改变重建。


In [ ]:
U,s,Vt = np.linalg.svd(X,full_matrices=False)
direction = Vt[0]
scores = X @ direction
reconstructed = scores[:,None]*direction+mean
fraction = s[0]**2/np.sum(s**2)
print('方向:',direction,'新坐标:',scores,'重建:',reconstructed,'保留方差:',fraction)
assert np.allclose(fraction,.9)
assert np.allclose(np.linalg.norm(data-reconstructed,'fro')**2,2)
fig,axes = plt.subplots(1,2,figsize=(10,4.5),dpi=150)
axes[0].scatter(*X.T,color='#2878b5',label='centered students')
line = np.outer(np.array([-3.,3.]),direction)
axes[0].plot(*line.T,color='#359969',label='first component')
for point,fit in zip(X,reconstructed-mean):
    axes[0].plot([point[0],fit[0]],[point[1],fit[1]],'--',color='#d97922',alpha=.7)
axes[0].scatter(*(reconstructed-mean).T,color='#c53351',marker='x',s=65,label='rank-one projection')
axes[0].set(xlim=(-3,3),ylim=(-3,3),aspect='equal',xlabel='test 1 deviation',ylabel='test 2 deviation',title='Direction and lost difference')
axes[0].legend(fontsize=8)
axes[1].bar([1,2],s**2/np.sum(s**2),color=['#2878b5','#d97922'])
axes[1].set(xticks=[1,2],xticklabels=['PC1','PC2'],ylim=(0,1),ylabel='variance fraction',title='90% vs 10%')
fig.tight_layout()
plt.show()


## 改变一列的刻度

把第二次成绩数值乘十，未标准化的主要方向怎样变化？注意不能直接比较不同刻度下的重建误差。


In [ ]:
scaled_data = data*np.array([1.,10.])
centered = scaled_data-scaled_data.mean(axis=0)
_,_,raw_Vt = np.linalg.svd(centered,full_matrices=False)
standard_deviation = centered.std(axis=0,ddof=1)
standardized = centered/standard_deviation
_,_,standard_Vt = np.linalg.svd(standardized,full_matrices=False)
print('改变刻度后的原始方向:',raw_Vt[0])
print('按列标准差缩放后的方向:',standard_Vt[0])
assert np.all(standard_deviation>0)


## 合上实验，自己解释

指出已知量、未知量、单位与计算目的。用纸上推导解释观察结果，再改变一个参数检查；有限次计算不是一般结论的证明。
